# Planet Sandbox: Analysis-Ready PlanetScope (Pará, Brazil) via Sentinel Hub

**Why the earlier notebook failed (404):** this sandbox collection (`byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e`, source `PS_ARD_SR_DAILY`) is hosted on **Sentinel Hub** (`services.sentinel-hub.com`), not on `api.planet.com`. Planet API keys only work with `api.planet.com`, and the `/data/collections/v2/...` path does not exist there.

**What this notebook does differently**
- Authenticates with an **OAuth client ID + secret** (not the API key).
- Searches with the Sentinel Hub **Catalog API**.
- Downloads a GeoTIFF with the **Process API**. There is no asset activation step.
- Then shows an RGB preview and NDVI.

**Before running (one time):**
1. Open the Planet Account app: <https://insights.planet.com/account/#/> → **OAuth Clients** → **Create New**.
2. Copy the **client ID** and **client secret** immediately (the secret is shown only once).
3. Your account needs Monitoring Credits (paid or trial). The sandbox data is CC-BY-NC.

The collection covers **2021-01-01 to 2023-12-29** over a 576 km² area centered near (-6.77, -52.38). Requests consume Processing Units from your account.

Never paste your secret into a shared notebook. It is requested with `getpass` below.

## 1. Install packages

In [42]:
#%pip install requests pandas rasterio matplotlib numpy

## 2. Imports

In [43]:
import json
import time
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests

## 3. Settings

In [44]:
COLLECTION_ID = "3f605f75-86c4-411a-b4ae-01c896f0e54e"
COLLECTION = f"byoc-{COLLECTION_ID}"

# Area of interest: [west, south, east, north] in EPSG:4326
BBOX = [-52.4825, -6.8735, -52.26725, -6.65775]

# Must fall inside 2021-01-01 .. 2023-12-29
START_DATE = "2013-01-01T00:00:00Z"
END_DATE = "2026-01-31T23:59:59Z"

# Output size in pixels (Process API maximum is 2500 x 2500).
# 2400 px over ~24 km is ~10 m per pixel. The native resolution is 3 m,
# so use a smaller BBOX if you want full resolution.
WIDTH = 2400
HEIGHT = 2400

# Sentinel Hub deployment. The Planet sandbox is normally on the EU one.
# If auth or search says the collection does not exist, try the US one:
#   "https://services-uswest2.sentinel-hub.com"
BASE_URL = "https://services.sentinel-hub.com"

print("Collection:", COLLECTION)
print("Date range:", START_DATE, "to", END_DATE)

Collection: byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e
Date range: 2013-01-01T00:00:00Z to 2026-01-31T23:59:59Z


## 4. Enter your OAuth client credentials (private)

In [45]:
CLIENT_ID = getpass("Planet OAuth client ID: ").strip()
CLIENT_SECRET = getpass("Planet OAuth client secret: ").strip()

if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Client ID and secret are both required.")

print("Credentials loaded.")

Credentials loaded.


## 5. Get an access token (cached and refreshed automatically)

In [46]:
_token = {"value": None, "expires_at": 0.0}


def get_token():
    """Return a valid bearer token, fetching a new one only when needed."""
    if _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]

    r = requests.post(
        f"{BASE_URL}/auth/realms/main/protocol/openid-connect/token",
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
        timeout=60,
    )

    if not r.ok:
        print("Token request failed:", r.status_code)
        print(r.text[:1000])
        r.raise_for_status()

    data = r.json()
    _token["value"] = data["access_token"]
    _token["expires_at"] = time.time() + int(data.get("expires_in", 3600))
    return _token["value"]


def auth_headers(**extra):
    return {"Authorization": f"Bearer {get_token()}", **extra}


get_token()
print("Authentication succeeded.")

Authentication succeeded.


## 6. Search the Catalog API
Lists the dates that have imagery over your AOI.

In [47]:
def catalog_search(body):
    # Try the documented catalog paths in turn (they are aliases of each other).
    last = None
    for path in ("/api/v1/catalog/1.0.0/search", "/catalog/v1/search"):
        r = requests.post(
            BASE_URL + path,
            headers=auth_headers(**{"Content-Type": "application/json"}),
            json=body,
            timeout=120,
        )
        if r.status_code != 404:
            return r
        last = r
    return last


body = {
    "collections": [COLLECTION],
    "bbox": BBOX,
    "datetime": f"{START_DATE}/{END_DATE}",
    "limit": 100,
}

features = []

for page in range(20):
    r = catalog_search(body)

    if not r.ok:
        print("HTTP status:", r.status_code)
        print(r.text[:2000])
        if r.status_code == 401:
            print("-> Credentials rejected. Check client ID/secret.")
        elif r.status_code == 403:
            print("-> Account may lack access/Monitoring Credits for this collection.")
        elif r.status_code == 404:
            print("-> Try the other deployment (BASE_URL in Cell 3).")
        r.raise_for_status()

    result = r.json()
    features.extend(result.get("features", []))

    nxt = result.get("context", {}).get("next")
    if not nxt:
        break
    body["next"] = nxt

print("Returned features:", len(features))

if not features:
    print("No scenes found. The collection covers 2021-01-01 to 2023-12-29; "
          "check START_DATE/END_DATE and that BBOX is inside the sandbox area.")

Returned features: 677


## 7. Inspect results

In [48]:
if not features:
    raise RuntimeError("No scenes found. See the message above.")

print("Properties available on the first feature:")
print(json.dumps(features[0].get("properties", {}), indent=2)[:1500])

rows = []
for f in features:
    p = f.get("properties", {})
    row = {"id": f.get("id"), "datetime": p.get("datetime")}
    for k, v in p.items():
        if "cloud" in k.lower():
            row[k] = v
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df["date"] = results_df["datetime"].str[:10]
display(results_df.head(30))

print("\nDates with data:", sorted(results_df["date"].unique()))

Properties available on the first feature:
{
  "datetime": "2023-12-29T00:00:00Z",
  "proj:epsg": 32722,
  "proj:bbox": [
    336000.0,
    9240000.0,
    360000.0,
    9264000.0
  ],
  "proj:geometry": {
    "type": "Polygon",
    "crs": {
      "type": "name",
      "properties": {
        "name": "urn:ogc:def:crs:EPSG::32722"
      }
    },
    "coordinates": [
      [
        [
          336000.00000084564,
          9240000.00000001
        ],
        [
          360000.00000031735,
          9240000.000000002
        ],
        [
          360000.0000003204,
          9264000.000000002
        ],
        [
          336000.00000085135,
          9264000.00000001
        ],
        [
          336000.00000084564,
          9240000.00000001
        ]
      ]
    ]
  }
}


,id,datetime,date
0,4fd3e6eb-1f6d-458a-bbae-79d66b583302,2023-12-29T00:00:00Z,2023-12-29
1,0792c687-c0f3-4c96-8080-1394cafe91ad,2023-12-28T00:00:00Z,2023-12-28
2,84298261-298a-46be-8ca3-b7ca814a48ea,2023-12-25T00:00:00Z,2023-12-25
3,b5658ba1-25c9-4f26-b057-ad096dab02a7,2023-12-20T00:00:00Z,2023-12-20
4,c521fc07-d467-486d-b95b-584856955cf1,2023-12-19T00:00:00Z,2023-12-19
5,4f92ec3f-df19-4c4d-a42d-d82aab42297c,2023-12-16T00:00:00Z,2023-12-16
6,e74712b2-5424-4115-b785-0bbb1833ec4b,2023-12-15T00:00:00Z,2023-12-15
7,4953a8f0-d1f7-4bd5-9b3d-d95d43ae2de5,2023-12-14T00:00:00Z,2023-12-14
8,8663ed59-1b2a-4c47-a181-49274d9fd961,2023-12-13T00:00:00Z,2023-12-13
9,48e44399-d92a-41c2-af78-e479a55a76bd,2023-12-08T00:00:00Z,2023-12-08



Dates with data: ['2021-01-01', '2021-01-02', '2021-01-05', '2021-01-06', '2021-01-07', '2021-01-08', '2021-01-18', '2021-01-22', '2021-01-26', '2021-01-27', '2021-02-01', '2021-02-02', '2021-02-03', '2021-02-07', '2021-02-24', '2021-03-05', '2021-03-15', '2021-03-16', '2021-03-20', '2021-03-21', '2021-03-23', '2021-03-24', '2021-03-26', '2021-03-27', '2021-03-29', '2021-03-30', '2021-04-03', '2021-04-04', '2021-04-07', '2021-04-08', '2021-04-09', '2021-04-11', '2021-04-12', '2021-04-13', '2021-04-14', '2021-04-15', '2021-04-16', '2021-04-24', '2021-04-25', '2021-04-26', '2021-04-29', '2021-05-02', '2021-05-03', '2021-05-04', '2021-05-06', '2021-05-08', '2021-05-10', '2021-05-11', '2021-05-12', '2021-05-13', '2021-05-15', '2021-05-16', '2021-05-17', '2021-05-20', '2021-05-21', '2021-05-22', '2021-05-23', '2021-05-24', '2021-05-25', '2021-05-26', '2021-05-27', '2021-05-28', '2021-05-29', '2021-05-30', '2021-05-31', '2021-06-01', '2021-06-02', '2021-06-03', '2021-06-04', '2021-06-05', '

In [49]:
results_df.to_csv("all_images_sentinel.csv", index=False)

## 8. Choose a date
By default this picks the date with the lowest cloud cover (if the catalog reports it), otherwise the first date. Override `SELECTED_DATE` to pick your own, e.g. `"2022-01-14"`.

In [50]:
SELECTED_DATE = None  # e.g. "2022-01-14"

cloud_cols = [c for c in results_df.columns if "cloud" in c.lower()]

if SELECTED_DATE is None:
    if cloud_cols:
        per_date = results_df.groupby("date")[cloud_cols[0]].mean().sort_values()
        SELECTED_DATE = per_date.index[0]
        print(f"Lowest mean {cloud_cols[0]}:", round(float(per_date.iloc[0]), 2))
    else:
        SELECTED_DATE = sorted(results_df["date"].unique())[0]

print("Selected date:", SELECTED_DATE)

Selected date: 2021-01-01


## 9. Download GeoTIFF with the Process API
Tries the standard 4-band surface-reflectance band names first, then the 8-band names. `dataMask` is added as an extra last band (1 = valid pixel).

In [51]:
""" 
CANDIDATE_BANDS = [
    ["blue", "green", "red", "nir"],
    ["blue", "green", "red", "nir", "rededge"],
    ["coastal_blue", "blue", "green_i", "green", "yellow", "red", "rededge", "nir"],
]


def build_evalscript(bands):
    inputs = ", ".join(f'"{b}"' for b in bands + ["dataMask"])
    returns = ", ".join(f"s.{b}" for b in bands + ["dataMask"])
#    return f### //VERSION=3
function setup() {{
  return {{
    input: [{{ bands: [{inputs}] }}],
    output: {{ bands: {len(bands) + 1}, sampleType: "FLOAT32" }}
  }};
}}
function evaluatePixel(s) {{
  return [{returns}];
}}
###


def process_request(bands):
    payload = {
        "input": {
            "bounds": {
                "bbox": BBOX,
                "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/4326"},
            },
            "data": [{
                "type": COLLECTION,
                "dataFilter": {
                    "timeRange": {
                        "from": f"{SELECTED_DATE}T00:00:00Z",
                        "to": f"{SELECTED_DATE}T23:59:59Z",
                    },
                    "mosaickingOrder": "mostRecent",
                },
            }],
        },
        "output": {
            "width": WIDTH,
            "height": HEIGHT,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": build_evalscript(bands),
    }
    return requests.post(
        f"{BASE_URL}/api/v1/process",
        headers=auth_headers(**{"Content-Type": "application/json", "Accept": "image/tiff"}),
        json=payload,
        timeout=600,
    )


OUTPUT_DIR = Path("planet_downloads")
OUTPUT_DIR.mkdir(exist_ok=True)
output_file = OUTPUT_DIR / f"ARPS_Para_{SELECTED_DATE}.tif"

used_bands = None

for bands in CANDIDATE_BANDS:
    print("Trying bands:", bands)
    r = process_request(bands)
    print("  HTTP status:", r.status_code)

    if r.ok:
        output_file.write_bytes(r.content)
        used_bands = bands
        break

    print("  ", r.text[:600])
    if r.status_code in (401, 403, 429):
        r.raise_for_status()

if used_bands is None:
    raise RuntimeError(
        "All band sets failed. Read the error text above: it usually names "
        "the invalid band or the access problem. Set CANDIDATE_BANDS to the "
        "band names it reports."
    )

print("Saved:", output_file)
print("File size:", f"{output_file.stat().st_size:,}", "bytes")
print("Bands in file:", used_bands + ["dataMask"])

"""

' \nCANDIDATE_BANDS = [\n    ["blue", "green", "red", "nir"],\n    ["blue", "green", "red", "nir", "rededge"],\n    ["coastal_blue", "blue", "green_i", "green", "yellow", "red", "rededge", "nir"],\n]\n\n\ndef build_evalscript(bands):\n    inputs = ", ".join(f\'"{b}"\' for b in bands + ["dataMask"])\n    returns = ", ".join(f"s.{b}" for b in bands + ["dataMask"])\n#    return f### //VERSION=3\nfunction setup() {{\n  return {{\n    input: [{{ bands: [{inputs}] }}],\n    output: {{ bands: {len(bands) + 1}, sampleType: "FLOAT32" }}\n  }};\n}}\nfunction evaluatePixel(s) {{\n  return [{returns}];\n}}\n###\n\n\ndef process_request(bands):\n    payload = {\n        "input": {\n            "bounds": {\n                "bbox": BBOX,\n                "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/4326"},\n            },\n            "data": [{\n                "type": COLLECTION,\n                "dataFilter": {\n                    "timeRange": {\n                        "from": f"

## 9.1. Download every 7th (date) GeoTIFF, always including the first and last

In [55]:
from pathlib import Path
from getpass import getpass
import time
import requests

# ============================================================
# 1. SETTINGS
# ============================================================

STEP = 7

COLLECTION_ID = "3f605f75-86c4-411a-b4ae-01c896f0e54e"
COLLECTION = f"byoc-{COLLECTION_ID}"

# Area of interest: west, south, east, north
# Coordinates are longitude/latitude in EPSG:4326
BBOX = [-52.4825, -6.8735, -52.26725, -6.65775]
BBOX_CRS = "http://www.opengis.net/def/crs/EPSG/0/4326"

START_DATE = "2013-01-01T00:00:00Z"
END_DATE = "2026-01-31T23:59:59Z"

BASE_URL = "https://services.sentinel-hub.com"

WIDTH = 1024
HEIGHT = 1024

OUTPUT_DIR = Path.cwd() / "planet_downloads"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the band names that worked in your previous Cell 9.
# If your earlier test used a different list, replace this list.
used_bands = ["blue", "green", "red", "nir"]

# ============================================================
# 2. CHECK CATALOG RESULTS
# ============================================================

if "results_df" not in globals():
    raise NameError(
        "results_df is missing. Run your Catalog search cell first."
    )

if len(results_df) == 0:
    raise ValueError("results_df is empty. No catalog results were found.")

all_dates = sorted(
    results_df["date"].astype(str).unique()
)

if len(all_dates) == 0:
    raise ValueError("No dates were found in results_df.")

# Select every STEP-th date
picked = all_dates[::STEP]

# Always include the last available acquisition date
if all_dates[-1] not in picked:
    picked.append(all_dates[-1])

print("Collection:", COLLECTION)
print("BBOX:", BBOX)
print("BBOX CRS:", BBOX_CRS)
print("Bands:", used_bands)
print(f"{len(all_dates)} dates found")
print(f"{len(picked)} dates selected")
print("Output folder:", OUTPUT_DIR.resolve())

# ============================================================
# 3. OAUTH AUTHENTICATION
# ============================================================

if "CLIENT_ID" not in globals() or not CLIENT_ID:
    CLIENT_ID = getpass("Planet OAuth client ID: ").strip()

if "CLIENT_SECRET" not in globals() or not CLIENT_SECRET:
    CLIENT_SECRET = getpass("Planet OAuth client secret: ").strip()

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        "CLIENT_ID and CLIENT_SECRET cannot be empty."
    )

TOKEN_URL = (
    f"{BASE_URL}/auth/realms/main/protocol/openid-connect/token"
)

_access_token = None
_token_expires_at = 0


def get_access_token():
    global _access_token, _token_expires_at

    response = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
        headers={
            "Content-Type": "application/x-www-form-urlencoded"
        },
        timeout=30,
    )

    if not response.ok:
        raise RuntimeError(
            "Token request failed\n"
            f"HTTP {response.status_code}\n"
            f"{response.text[:500]}"
        )

    token_data = response.json()

    _access_token = token_data["access_token"]

    expires_in = token_data.get("expires_in", 3600)
    _token_expires_at = time.time() + expires_in - 60

    return _access_token


def auth_headers():
    if (
        _access_token is None
        or time.time() >= _token_expires_at
    ):
        get_access_token()

    return {
        "Authorization": f"Bearer {_access_token}",
        "Content-Type": "application/json",
        "Accept": "image/tiff",
    }


# Test authentication now
get_access_token()
print("Authentication successful.")

# ============================================================
# 4. EVALSCRIPT
# ============================================================

def build_evalscript(bands):
    input_bands = list(bands) + ["dataMask"]

    input_text = ", ".join(
        f'"{band}"' for band in input_bands
    )

    output_text = ", ".join(
        f"sample.{band}" for band in input_bands
    )

    return f"""//VERSION=3

function setup() {{
  return {{
    input: [{{
      bands: [{input_text}]
    }}],
    output: {{
      bands: {len(input_bands)},
      sampleType: "FLOAT32"
    }}
  }};
}}

function evaluatePixel(sample) {{
  return [{output_text}];
}}
"""

# ============================================================
# 5. CREATE PROCESSING API REQUEST
# ============================================================

def process_request(date_string):
    payload = {
        "input": {
            "bounds": {
                "bbox": BBOX,
                "properties": {
                    "crs": BBOX_CRS
                }
            },
            "data": [
                {
                    "type": COLLECTION,
                    "dataFilter": {
                        "timeRange": {
                            "from": f"{date_string}T00:00:00Z",
                            "to": f"{date_string}T23:59:59Z"
                        },
                        "mosaickingOrder": "mostRecent"
                    }
                }
            ]
        },
        "output": {
            "width": WIDTH,
            "height": HEIGHT,
            "responses": [
                {
                    "identifier": "default",
                    "format": {
                        "type": "image/tiff"
                    }
                }
            ]
        },
        "evalscript": build_evalscript(used_bands)
    }

    return requests.post(
        f"{BASE_URL}/api/v1/process",
        headers=auth_headers(),
        json=payload,
        timeout=600,
    )

# ============================================================
# 6. DOWNLOAD SELECTED DATES
# ============================================================

downloaded = []
failed = []

for date_string in picked:
    output_path = (
        OUTPUT_DIR / f"ARPS_Para_{date_string}.tif"
    )

    # Do not download an existing non-empty file again
    if (
        output_path.exists()
        and output_path.stat().st_size > 0
    ):
        print(date_string, "- already downloaded, skipping")
        downloaded.append(output_path)
        continue

    try:
        response = process_request(date_string)

        if response.ok:
            output_path.write_bytes(response.content)
            downloaded.append(output_path)

            print(
                date_string,
                "- saved",
                f"{output_path.stat().st_size:,}",
                "bytes",
            )

        else:
            failed.append(date_string)

            print(
                date_string,
                "- FAILED",
                response.status_code,
                response.text[:500],
            )

            # Stop for authentication, permission, or quota errors
            if response.status_code in (401, 403, 429):
                print(
                    "Stopping because of authentication, "
                    "permission, or rate-limit error."
                )
                break

    except Exception as error:
        failed.append(date_string)

        print(
            date_string,
            "- ERROR:",
            repr(error),
        )

        break

    time.sleep(1)

# ============================================================
# 7. SUMMARY
# ============================================================

print()
print("Done")
print("Downloaded:", len(downloaded))
print("Failed:", len(failed))
print("Bands used:", used_bands)
print("Files saved in:", OUTPUT_DIR.resolve())

if failed:
    print("Failed dates:", failed)


Collection: byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e
BBOX: [-52.4825, -6.8735, -52.26725, -6.65775]
BBOX CRS: http://www.opengis.net/def/crs/EPSG/0/4326
Bands: ['blue', 'green', 'red', 'nir']
677 dates found
98 dates selected
Output folder: /Users/huss/data-processing/Github/TechnicalEvaluation-3-planet-dashboard-app-Solutions-Architect/Notebook/planet_downloads
Authentication successful.
2021-01-01 - already downloaded, skipping
2021-01-22 - already downloaded, skipping
2021-02-24 - already downloaded, skipping
2021-03-24 - already downloaded, skipping
2021-04-07 - already downloaded, skipping
2021-04-15 - already downloaded, skipping
2021-05-03 - already downloaded, skipping
2021-05-13 - already downloaded, skipping
2021-05-23 - already downloaded, skipping
2021-05-30 - already downloaded, skipping
2021-06-06 - already downloaded, skipping
2021-06-13 - already downloaded, skipping
2021-06-21 - already downloaded, skipping
2021-06-28 - already downloaded, skipping
2021-07-06 - alread